# DSP501: Noise-Robust Speaker Verification

This notebook is a technical dashboard for the versioned DSP501 study. [`run.py`](run.py) is the canonical executor; this notebook invokes its CLI and displays saved artifacts without reimplementing the audio pipeline, training, threshold calibration, evaluation, or plotting. [`report.qmd`](report.qmd) is the final Quarto delivery document.

## Research questions, hypothesis, and scope

**RQ1.** How does deterministic composite MUSAN noise affect text-independent ECAPA-TDNN speaker verification across 5, 10, 15, and 20 dB test SNR?

**RQ2.** Relative to the raw-noisy control, do progressive high-pass, high-pass/low-pass, and Wiener-enhanced DSP stages improve verification? **H1:** the full offline DSP chain improves F1 and/or EER at low SNR relative to raw-noisy audio.

Pipeline: VCTK 0.92 source speech → deterministic composite MUSAN mixing → optional zero-phase filters and Wiener stage → mono 16 kHz ECAPA-TDNN → cosine speaker-verification scores. The 16 kHz model input has an 8 kHz Nyquist limit. The filters are zero-phase, so this is an offline analysis method, not a streaming-deployment claim.

## Fixed experimental protocol

The versioned configuration fixes 3-second clips, 50 deterministic clips per speaker, speaker-disjoint 80/10/10 train/validation/test splits, seeds 11/22/33, and test SNRs 5/10/15/20 dB. Validation selects each threshold; test metrics always use the locked validation threshold. The five conditions are `clean_reference`, `raw_noisy`, `high_pass`, `high_pass_low_pass`, and `full_dsp`.

`clean_reference` is a clean-test ceiling/reference, not a head-to-head noisy comparison. Tuning is performed only for `raw_noisy` and locks learning rate and encoder-freeze epochs for all subsequent ablations. The DSP-only comparator is an MFCC mean/std embedding scored with cosine similarity.

## Environment, configuration, and data-license gate

Run `uv sync` before opening this notebook. Downloading VCTK and MUSAN is opt-in: read and accept both dataset licenses before changing the gate below. The full-study command does not download data.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "dsp501-v1.json"
if not (PROJECT_ROOT / "pyproject.toml").is_file() or not CONFIG_PATH.is_file():
    raise RuntimeError("Open main.ipynb from the DSP project root.")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
STUDY_ROOT = PROJECT_ROOT / config["output_root"] / config["study_id"]
REPORT_ROOT = STUDY_ROOT / "reports"
print(json.dumps(config, indent=2))

{
  "description": "DSP501 noise-robust text-independent speaker-verification study.",
  "study_id": "dsp501-speaker-verification-v1",
  "config_version": 1,
  "condition": "raw_noisy",
  "experimental_seeds": [
    11,
    22,
    33
  ],
  "segment_seconds": 3.0,
  "clips_per_speaker": 50,
  "positive_pairs_per_speaker": 50,
  "snr_db": [
    5,
    10,
    15,
    20
  ],
  "high_pass_hz": 80.0,
  "low_pass_hz": 7500.0,
  "wiener_window_size": 29,
  "ecapa_source": "speechbrain/spkrec-ecapa-voxceleb",
  "ecapa_revision": "0f99f2d0ebe89ac095bcc5903c4dd8f72b367286",
  "vctk_root": "dataset/VCTK-Corpus-0.92/wav48_silence_trimmed",
  "musan_root": "dataset/musan",
  "output_root": "outputs",
  "ecapa_cache": "pretrained_models/spkrec-ecapa-voxceleb"
}


In [ ]:
# Change to True only after you have read and accepted the VCTK and MUSAN licenses.
ACCEPT_DATA_LICENSES = True
if not ACCEPT_DATA_LICENSES:
    raise PermissionError(
        "Data download is gated. Review the VCTK and MUSAN licenses, then set ACCEPT_DATA_LICENSES = True."
    )

required_data = {
    "VCTK wav48_silence_trimmed": PROJECT_ROOT / config["vctk_root"],
    "MUSAN": PROJECT_ROOT / config["musan_root"],
}
missing = {name: path for name, path in required_data.items() if not path.is_dir()}
if missing:
    raise FileNotFoundError(
        "Missing data directories:\n"
        + "\n".join(f"- {name}: {path}" for name, path in missing.items())
    )
display(Markdown("**Data preflight passed.**"))

FileNotFoundError: Missing data directories:
- VCTK wav48_silence_trimmed: /home/truong51972/projects/dsp/dataset/VCTK-Corpus-0.92/wav48_silence_trimmed
- MUSAN: /home/truong51972/projects/dsp/dataset/musan

## Canonical CLI execution

The following cell is intentionally expensive. It runs prepare, tuning, training, evaluation, analysis, and release packaging in the supported canonical order. Run individual stages below only for recovery or debugging; do not replace these commands with notebook-side study logic.

In [ ]:
# INTENTIONALLY EXPENSIVE: requires downloaded data, model access/cache, and substantial compute.
!uv run python run.py all

In [ ]:
# Download only after the explicit license gate above is set to True.
!uv run python run.py --config configs/dsp501-v1.json download-data --accept-data-licenses

In [ ]:
# Individual supported stages for recovery/debugging.
!uv run python run.py --config configs/dsp501-v1.json prepare
!uv run python run.py --config configs/dsp501-v1.json tune
!uv run python run.py --config configs/dsp501-v1.json train
!uv run python run.py --config configs/dsp501-v1.json evaluate
!uv run python run.py --config configs/dsp501-v1.json analyze
!uv run python run.py --config configs/dsp501-v1.json package

## Saved-artifact dashboard

The cells below are read-only viewers. They fail with a useful message until `analyze` has written the expected reports. They never recompute a metric or generate a figure.

In [ ]:
def require_artifact(relative_path: str) -> Path:
    path = STUDY_ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing artifact: {path}. Run `uv run python run.py analyze` after evaluation."
        )
    return path


def show_saved_image(filename: str) -> None:
    display(Image(filename=str(require_artifact(f"reports/{filename}"))))

In [ ]:
# Locked raw-noisy hyperparameters and every validation-EER trial.
tuning = json.loads(require_artifact("tuning.json").read_text(encoding="utf-8"))
display(Markdown("### Tuning: locked hyperparameters"))
display(tuning)

In [ ]:
# Machine-readable metric summary written by the analysis stage.
with require_artifact("reports/metric-summary.csv").open(
    newline="", encoding="utf-8"
) as stream:
    metric_rows = list(csv.DictReader(stream))
display(Markdown(f"### Verification metrics ({len(metric_rows)} saved rows)"))
display(metric_rows[:20])

In [ ]:
# Saved ablation, SNR, and 5 dB confusion-matrix figures.
show_saved_image("ablation-f1.png")
show_saved_image("snr-curves.png")
show_saved_image("confusion-matrices.png")

In [ ]:
# Paired, stratified-bootstrap 95% CIs for full-DSP minus raw-noisy F1.
bootstrap_intervals = json.loads(
    require_artifact("reports/paired-bootstrap-ci.json").read_text(encoding="utf-8")
)
display(Markdown("### Paired bootstrap confidence intervals"))
display(bootstrap_intervals)

In [ ]:
# Retained false-positive/false-negative provenance, score margins, residual SNR, and MFCC diagnostics.
error_reports = sorted(STUDY_ROOT.glob("seed-*/*/evaluation-test-snr-*.json"))
if not error_reports:
    raise FileNotFoundError(
        f"No saved test evaluation reports found in {STUDY_ROOT}. Run `evaluate` and `analyze`."
    )
saved_errors = []
for report in error_reports:
    payload = json.loads(report.read_text(encoding="utf-8"))
    saved_errors.extend(
        {**error, "report": str(report.relative_to(PROJECT_ROOT))}
        for error in payload.get("errors", [])
    )
display(Markdown(f"### Saved error cases ({len(saved_errors)})"))
display(saved_errors[:20])

In [ ]:
# Signal diagnostics created by the analysis stage for one deterministic held-out clip.
show_saved_image("waveforms.png")
show_saved_image("psd-spectrogram.png")
show_saved_image("mfcc.png")
show_saved_image("filter-response.png")

## Release bundle and reproducibility

The package command excludes raw datasets and writes a SHA-256 manifest. Verify the staged bundle after a full run; use the report as the final renderable presentation layer.

In [ ]:
!uv run python run.py --config configs/dsp501-v1.json package --dry-run
!uv run python run.py --config configs/dsp501-v1.json package
!uv run pytest -q
!quarto render report.qmd